# Regulators and signalling footprints along the PT (notebook 95)

**The question.** Pathway methods so far score a pathway by its *members*. Footprint methods score a regulator or a
signalling pathway by its *targets*. Does target-based activity (CollecTRI regulons, PROGENy footprints) give PT
biology that is specific, replicated between animals and confirmed in independent data — for healthy zonation in each
species, the species comparison and injury — and does it beat member-based sets scored the same way?

**The frozen protocol** is `docs/results/pt-regulator-activity.md` (commit d64bfd1), fixed before any activity was
computed. One estimator for every family: decoupleR's univariate linear model (ulm), z against 1,000 permutations of
the statistic within expression deciles.

**Sections.** 1 setup and guards; 2 networks and families; 3 zonation; 4 injury; 5 scorecard and verdicts;
6 PROGENy against its named sets; 7 recovery checks and mRNA; 8 figures; 9 reading and run record.

## 1 · Setup

In [ ]:
import argparse
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

NOTEBOOK_START = time.perf_counter()
starts = [Path.cwd()] + ([Path(__file__).resolve().parent] if '__file__' in globals() else [])
project = next(p for s in starts for p in (s, *s.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
NETS = data_root / 'external' / 'regulator_networks'
CENSUS = data_root / 'external' / 'census_pt_segments'
T71 = results_root / 'pt_pathways_package_pseudospace' / 'tables'
T90 = results_root / 'pt_continuous_axis_experiments' / 'tables'
T91 = results_root / 'pt_positional_decomposition' / 'tables'
R94 = results_root / 'pt_external_iri'
ORTHOLOGS = results_root / 'minimal_pt_scfates' / 'ortholog_map_used.csv'
output = results_root / 'pt_regulator_activity'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)

NOTEBOOK_LOGIC_VERSION = '95.regulators.1'
SEED = 95
ALPHA = .05
MIN_TARGETS = 10
N_PERM = 1000
PROGENY_TOP = 500
RELABEL_RATIO = .5
MIN_CELLS = 50
MIN_CALLS = 5
TOL = {'between-mouse ρ': .05, 'mouse external enrichment': .25, 'human external enrichment': .25,
       'specificity': .05, 'pair agreement': .05, 'external enrichment': .25}
DIRECTIONS = {'between-mouse ρ': 1, 'mouse external enrichment': 1, 'human external enrichment': 1,
              'specificity': -1, 'pair agreement': 1, 'external enrichment': 1}
LAYER_CRITERIA = {'Z': ['between-mouse ρ', 'mouse external enrichment', 'human external enrichment'],
                  'I': ['specificity', 'pair agreement', 'external enrichment']}
COMPARISONS = [('REG', 'MEM'), ('REG', 'HALL'), ('FOOT', 'HALL'), ('FOOT', 'MEM'), ('HALL', 'MEM')]
FAMILIES = ['REG', 'FOOT', 'HALL', 'MEM']
MOUSE_SPEC, HUMAN_SPEC = ['Ctrl1A2', 'Ctrl1A4'], ['HUK1_COR1', 'HUK1_MED1']
PAIRS = ['IR2A2-Ctrl1A2', 'IR2A2-Ctrl1A4', 'IR2A4-Ctrl1A2', 'IR2A4-Ctrl1A4']
TIMES = ['4hours', '12hours', '2days', '14days', '6weeks']
T_LABEL = {'4hours': '4 h', '12hours': '12 h', '2days': '2 d', '14days': '14 d', '6weeks': '6 wk'}
FOOT_SETS = {  # PROGENy pathway -> (Hallmark, KEGG), frozen in the protocol
    'TNFa': ('TNF-alpha Signaling via NF-kB', 'TNF signaling pathway'), 'NFkB': (None, 'NF-kappa B signaling pathway'),
    'p53': ('p53 Pathway', 'p53 signaling pathway'), 'Hypoxia': ('Hypoxia', 'HIF-1 signaling pathway'),
    'TGFb': ('TGF-beta Signaling', 'TGF-beta signaling pathway'), 'JAK-STAT': ('IL-6/JAK/STAT3 Signaling', 'JAK-STAT signaling pathway'),
    'EGFR': (None, 'ErbB signaling pathway'), 'MAPK': (None, 'MAPK signaling pathway'),
    'PI3K': ('PI3K/AKT/mTOR  Signaling', 'PI3K-Akt signaling pathway'), 'VEGF': (None, 'VEGF signaling pathway'),
    'WNT': ('Wnt-beta Catenin Signaling', 'Wnt signaling pathway'), 'Estrogen': ('Estrogen Response Early', 'Estrogen signaling pathway'),
    'Androgen': ('Androgen Response', None), 'Trail': (None, 'Apoptosis')}
RECOVERY = [('RC1', 'HNF4A', -1), ('RC1', 'HNF1B', -1), ('RC1', 'PBX1', -1), ('RC2', 'NFKB', 1), ('RC2', 'RELB', 1),
            ('RC3', 'TCF7L1', 1), ('RC4', 'MYC', 1)]

REQUIRED = [NETS / 'collectri_human.tsv', NETS / 'progeny_human.tsv', ORTHOLOGS, T71 / 'gene_sets_aki_vs_control.json',
            T90 / 'd_end_contrasts.csv.gz', T91 / 'i_genes.csv.gz', R94 / 'run_record.json',
            R94 / 'tables' / 'external_genes.csv.gz', R94 / 'tables' / 'pseudobulk_counts.csv.gz',
            CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', CENSUS / 'human_pt_segment_pseudobulk_meta.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (download the networks; run notebooks 71, 90, 91 and 94 first)')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom, norm, spearmanr
from IPython.display import display

from pseudospace.pathway_networks import dominance
from pseudospace.regulator_activity import (collectri_network, progeny_network, stratified_permutations, ulm_null,
                                            weight_matrix)
from pseudospace.stats_gam import bh_adjust

MM = 1 / 25.4
INK, MUTED, LIGHT = '#0b0b0b', '#52514e', '#d9d8d4'
FAMILY_COLOR = {'REG': '#d55e00', 'FOOT': '#9C93D3', 'HALL': '#2a78d6', 'MEM': '#1b8a5a'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_colwidth', 70, 'display.max_rows', 80)


def save(fig, stem):
    for suffix in ('png', 'pdf'):
        fig.savefig(figures / f'{stem}.{suffix}', bbox_inches='tight')

### 1.1 · Guards (not cached)

- G1: the network files hold the frozen structure (64,979 CollecTRI edges from 1,209 sources; 14 PROGENy pathways);
  their sha256 is recorded.
- G2: notebook 94 finished and recorded its t*.

In [ ]:
guard_log = []
SHA = {f: hashlib.sha256((NETS / f).read_bytes()).hexdigest() for f in ('collectri_human.tsv', 'progeny_human.tsv')}
ct_rows = pd.read_csv(NETS / 'collectri_human.tsv', sep='\t',
                      usecols=['source_genesymbol', 'target_genesymbol', 'is_stimulation', 'is_inhibition'])
pg_rows = pd.read_csv(NETS / 'progeny_human.tsv', sep='\t')
assert len(ct_rows) == 64979 and ct_rows.source_genesymbol.nunique() == 1209, 'G1: CollecTRI'
assert pg_rows[pg_rows.label.eq('pathway')].value.nunique() == 14, 'G1: PROGENy'
guard_log.append(f"G1: CollecTRI 64,979 edges / 1,209 sources, PROGENy 14 pathways (sha256 {SHA['collectri_human.tsv'][:12]}…, "
                 f"{SHA['progeny_human.tsv'][:12]}…)")
RUN94 = json.loads((R94 / 'run_record.json').read_text())
T_STAR = {v: k for k, v in T_LABEL.items()}[RUN94['t_star']]
guard_log.append(f"G2: notebook 94's t* is {RUN94['t_star']}")
print('\n'.join('· ' + g for g in guard_log))

## 2 · Networks and families

In [ ]:
orth = pd.read_csv(ORTHOLOGS)
h2m = dict(zip(orth.human_symbol.astype(str), orth.mouse_symbol.astype(str)))
REG_NET = collectri_network(ct_rows, h2m)
FOOT_NET = progeny_network(pg_rows, h2m, top=PROGENY_TOP)
SETS71 = json.loads((T71 / 'gene_sets_aki_vs_control.json').read_text())


def set_net(prefixes):
    rows = [(k.split('::', 1)[1], g, 1.) for k, v in SETS71.items() if k.split('::', 1)[0] in prefixes for g in v]
    return pd.DataFrame(rows, columns=['source', 'target', 'weight']).drop_duplicates(['source', 'target'])


NETWORKS = {'REG': REG_NET, 'FOOT': FOOT_NET, 'HALL': set_net({'MSigDB_Hallmark_2020'}),
            'MEM': set_net({'KEGG_2019_Mouse', 'Reactome_2022'})}
print({k: (v.source.nunique(), len(v)) for k, v in NETWORKS.items()})

G91 = pd.read_csv(T91 / 'i_genes.csv.gz').set_index('gene')
G91.index = G91.index.astype(str)
EXPR = G91['log2 CP10k (controls)']


def strata_for(genes):
    """Expression deciles over ``genes`` (91's control log2 CP10k); genes without it form their own stratum."""
    e = EXPR.reindex(genes)
    out = np.full(len(genes), -1)
    ok = e.notna().to_numpy()
    out[ok] = np.minimum((e[ok].rank(pct=True).to_numpy() * 10).astype(int), 9)
    return out


def activity(family, genes, vectors, seed):
    """{vector: DataFrame(source, t, z, p, q)} for one family over ``genes`` (vectors: name -> Series over genes)."""
    W, sources = weight_matrix(NETWORKS[family], genes, min_targets=MIN_TARGETS)
    perms = stratified_permutations(strata_for(genes), N_PERM, seed)
    out = {}
    for name, v in vectors.items():
        t, z = ulm_null(W, v.reindex(genes).to_numpy(float), perms)
        p = 2 * norm.sf(np.abs(z))
        out[name] = pd.DataFrame({'t': t, 'z': z, 'p': p, 'q': bh_adjust(p)}, index=pd.Index(sources, name='source'))
    return out, sources


SOURCES = []

## 3 · Z · Healthy zonation

Vectors: notebook 90's end contrasts E per specimen (mean of the last 30% − first 30% of its own curve), on its
7,971 genes. Positional in a species: q ≤ 0.05 in both of its specimens, same sign. External: Kirita's sham mice
(mean over mice of S3 − S1 log2 CPM) and Census human donors (segment 3 − convoluted).

In [ ]:
END = pd.read_csv(T90 / 'd_end_contrasts.csv.gz').set_index('mouse')
END.index = END.index.astype(str)
END = END[np.isfinite(END[[f'E {s}' for s in MOUSE_SPEC + HUMAN_SPEC]]).all(axis=1)]
ZG = END.index.tolist()

pb = pd.read_csv(R94 / 'tables' / 'pseudobulk_counts.csv.gz', index_col=0)
pb.index = pb.index.astype(str)
sham = sorted({c.rsplit('|', 1)[0] for c in pb.columns if c.startswith('Control|')})
ext94 = pd.read_csv(R94 / 'tables' / 'external_genes.csv.gz', header=[0, 1], index_col=0)
ext94.index = ext94.index.astype(str)
E94 = ext94.index.tolist()


def lcpm(col):
    return np.log2((pb[col] + .5) / (pb[col].sum() + 1) * 1e6)


kirita_zon = pd.concat([lcpm(f'{m}|PTS3') - lcpm(f'{m}|PTS1') for m in sham], axis=1).mean(axis=1)
cmeta = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_meta.csv', index_col=0)
ccounts = pd.read_csv(CENSUS / 'human_pt_segment_pseudobulk_counts.csv.gz', index_col=0)
S3_TYPE, PCT_TYPE = 'epithelial cell of proximal tubule segment 3', 'kidney proximal convoluted tubule epithelial cell'
donors = [d for d, b in cmeta.groupby('donor_id')
          if {S3_TYPE, PCT_TYPE} <= set(b.cell_type[b.n_cells >= MIN_CELLS])]
census_diff = []
for d in donors:
    b = cmeta[cmeta.donor_id.eq(d)]
    vals = {}
    for ct in (S3_TYPE, PCT_TYPE):
        rows = b.index[b.cell_type.eq(ct)]
        y = ccounts.loc[rows].sum(axis=0)
        vals[ct] = np.log2((y + .5) / (b.loc[rows, 'total_counts_all_genes'].sum() + 1) * 1e6)
    census_diff.append(vals[S3_TYPE] - vals[PCT_TYPE])
census_zon = pd.concat(census_diff, axis=1).mean(axis=1)
census_zon.index = census_zon.index.astype(str)
print(f'Z genes {len(ZG):,}; Kirita sham mice {len(sham)}; Census donors {len(donors)}')

ZONE = {}
for fam in FAMILIES:
    ours, src = activity(fam, ZG, {s: END[f'E {s}'] for s in MOUSE_SPEC + HUMAN_SPEC}, SEED)
    zk = [g for g in ZG if g in set(E94) and g in pb.index]
    kir, src_k = activity(fam, zk, {'Kirita sham': kirita_zon}, SEED + 1)
    zc = [g for g in ZG if g in census_zon.index]
    cen, src_c = activity(fam, zc, {'Census human': census_zon}, SEED + 2)
    ZONE[fam] = {**ours, **kir, **cen}
    SOURCES.append({'family': fam, 'layer': 'Z', 'sources': len(src), 'sources (Kirita genes)': len(src_k),
                    'sources (Census genes)': len(src_c)})


def positional(frames, specimens):
    a, b = (frames[s] for s in specimens)
    ok = (a.q <= ALPHA) & (b.q <= ALPHA) & (np.sign(a.z) == np.sign(b.z))
    return pd.Series(np.where(ok, np.sign(a.z), 0.), index=a.index)


ZROWS = []
for fam in FAMILIES:
    f = ZONE[fam]
    pm, ph = positional(f, MOUSE_SPEC), positional(f, HUMAN_SPEC)
    cat = np.select([(pm != 0) & (ph != 0) & (pm == ph), (pm != 0) & (ph != 0) & (pm != ph), (pm != 0), (ph != 0)],
                    ['conserved', 'reversed', 'mouse only', 'human only'], 'none')
    table = pd.DataFrame({**{f'z {s}': f[s].z for s in MOUSE_SPEC + HUMAN_SPEC},
                          **{f'q {s}': f[s].q for s in MOUSE_SPEC + HUMAN_SPEC},
                          'mouse positional': pm, 'human positional': ph, 'category': cat}, index=f[MOUSE_SPEC[0]].index)
    for ext in ('Kirita sham', 'Census human'):
        table[f'z {ext}'] = f[ext].z.reindex(table.index)
        table[f'q {ext}'] = f[ext].q.reindex(table.index)
    ZROWS.append(table.assign(family=fam))
ZTAB = pd.concat(ZROWS).reset_index()
ZTAB.to_csv(tables / 'activity_zonation.csv', index=False)
display(ZTAB.groupby(['family', 'category']).size().unstack(fill_value=0))

## 4 · I · Injury level (and location, descriptive)

Vectors: notebook 91's L (condition), the four IR × control pair L, L R1 and L R2, on its 10,361 shared genes.
Called: q ≤ 0.05 on L; all four pair t share its sign; |t_R| ≤ 0.5 |t| in both relabelings. External: Kirita's level
t at every time point (94's universe ∩ these genes). Location (descriptive): Δ late − L against Kirita's location t.

In [ ]:
IG = G91.index.tolist()
ivec = {'condition': G91.L, **{p: G91[f'L {p}'] for p in PAIRS}, 'R1': G91['L R1'], 'R2': G91['L R2'],
        'location': G91['Δ late'] - G91.L}
IE = [g for g in IG if g in set(E94)]
INJ, IROWS = {}, []
for fam in FAMILIES:
    ours, src = activity(fam, IG, ivec, SEED + 3)
    ext = {f'Kirita level {T_LABEL[t]}': ext94[(t, 'level t')] for t in TIMES}
    ext.update({f'Kirita location {T_LABEL[t]}': ext94[(t, 'location t')] for t in TIMES})
    kir, src_k = activity(fam, IE, ext, SEED + 4)
    INJ[fam] = {**ours, **kir}
    SOURCES.append({'family': fam, 'layer': 'I', 'sources': len(src), 'sources (Kirita genes)': len(src_k)})
    c = ours['condition']
    pair_ok = np.all([np.sign(ours[p].t) == np.sign(c.t) for p in PAIRS], axis=0)
    relab_ok = (ours['R1'].t.abs() <= RELABEL_RATIO * c.t.abs()) & (ours['R2'].t.abs() <= RELABEL_RATIO * c.t.abs())
    called = (c.q <= ALPHA) & pair_ok & relab_ok
    table = pd.DataFrame({'t': c.t, 'z': c.z, 'q': c.q, 'pairs agree': pair_ok, 'relabelings small': relab_ok,
                          'called': called, 'z R1': ours['R1'].z, 'q R1': ours['R1'].q, 'z R2': ours['R2'].z,
                          'q R2': ours['R2'].q, 'z location': ours['location'].z, 'q location': ours['location'].q},
                         index=c.index)
    for name, frame in kir.items():
        table[f'z {name}'] = frame.z.reindex(table.index)
        table[f'q {name}'] = frame.q.reindex(table.index)
    IROWS.append(table.assign(family=fam))
ITAB = pd.concat(IROWS).reset_index()
ITAB.to_csv(tables / 'activity_injury.csv', index=False)
display(ITAB.groupby('family')[['called']].sum().assign(
    **{'q ≤ 0.05': ITAB.groupby('family').q.apply(lambda q: int((q <= ALPHA).sum())),
       'R1 q ≤ 0.05': ITAB.groupby('family')['q R1'].apply(lambda q: int((q <= ALPHA).sum())),
       'R2 q ≤ 0.05': ITAB.groupby('family')['q R2'].apply(lambda q: int((q <= ALPHA).sum()))}))
SOURCE_TABLE = pd.DataFrame(SOURCES)
SOURCE_TABLE.to_csv(tables / 'sources.csv', index=False)
display(SOURCE_TABLE)

## 5 · Scorecard and verdicts

In [ ]:
def enrichment(called, replicated):
    """(replicated share of called) / (replicated share of all), and the hypergeometric p, over scorable sources."""
    called, replicated = np.asarray(called, bool), np.asarray(replicated, bool)
    n, k, N, K = int(called.sum()), int((called & replicated).sum()), len(called), int(replicated.sum())
    if n == 0 or K == 0:
        return np.nan, np.nan, n, k
    return (k / n) / (K / N), float(hypergeom.sf(k - 1, N, K, n)), n, k


SCORE, CALLS = [], []
t_star = f'Kirita level {T_LABEL[T_STAR]}'
for fam in FAMILIES:
    z = ZTAB[ZTAB.family.eq(fam)].set_index('source')
    row = {'family': fam}
    row['between-mouse ρ'] = spearmanr(z[f'z {MOUSE_SPEC[0]}'], z[f'z {MOUSE_SPEC[1]}'])[0]
    for label, ext, pos in (('mouse', 'Kirita sham', 'mouse positional'), ('human', 'Census human', 'human positional')):
        ok = z[f'z {ext}'].notna()
        rep = (z[f'q {ext}'] <= ALPHA) & (np.sign(z[f'z {ext}']) == np.sign(z[pos]))
        e, p, n, k = enrichment((z[pos] != 0)[ok], rep[ok])
        row.update({f'{label} external enrichment': e if n >= MIN_CALLS else np.nan, f'{label} external p': p,
                    f'{label} positional (scorable)': n, f'{label} replicated': k})
    i = ITAB[ITAB.family.eq(fam)].set_index('source')
    n_called = int(i.called.sum())
    relab = np.mean([(i['q R1'] <= ALPHA).sum(), (i['q R2'] <= ALPHA).sum()])
    sig = i.q <= ALPHA
    row['specificity'] = relab / max(int(sig.sum()), 1) if sig.sum() >= MIN_CALLS else np.nan
    row['pair agreement'] = float(i['pairs agree'][sig].mean()) if sig.sum() >= MIN_CALLS else np.nan
    ok = i[f'z {t_star}'].notna()
    rep = (i[f'q {t_star}'] <= ALPHA) & (np.sign(i[f'z {t_star}']) == np.sign(i.z))
    e, p, n, k = enrichment(i.called[ok], rep[ok])
    row.update({'external enrichment': e if n >= MIN_CALLS else np.nan, 'external p': p, 'called (scorable)': n,
                'called replicated': k, 'called': n_called, 'q ≤ 0.05': int(sig.sum()), 'relabeled q ≤ 0.05 (mean)': relab})
    SCORE.append(row)
SCORE = pd.DataFrame(SCORE).set_index('family')
SCORE.to_csv(tables / 'scorecard.csv')
display(SCORE.round(3).T)

VERD = []
for a, b in COMPARISONS:
    for layer, crit in LAYER_CRITERIA.items():
        va = {c: SCORE.loc[a, c] for c in crit}
        vb = {c: SCORE.loc[b, c] for c in crit}
        VERD.append({'A': a, 'B': b, 'layer': layer, 'verdict': dominance(va, vb, DIRECTIONS, TOL),
                     **{f'{c} (A, B)': f'{va[c]:.3g}, {vb[c]:.3g}' for c in crit}})
VERD = pd.DataFrame(VERD)
VERD.to_csv(tables / 'verdicts.csv', index=False)
display(VERD)

CALLS = pd.concat([ZTAB[ZTAB.category.ne('none')].assign(layer='Z')[['family', 'source', 'layer', 'category']],
                   ITAB[ITAB.called].assign(layer='I', category=np.where(ITAB.z[ITAB.called] > 0, 'up', 'down'))
                   [['family', 'source', 'layer', 'category']]])
CALLS.to_csv(tables / 'calls.csv', index=False)
for fam in ('REG', 'FOOT'):
    z = ZTAB[ZTAB.family.eq(fam)]
    print(fam, 'conserved:', ', '.join(z.source[z.category.eq('conserved')].head(40)))
    print(fam, 'reversed:', ', '.join(z.source[z.category.eq('reversed')].head(40)))
    i = ITAB[ITAB.family.eq(fam) & ITAB.called].sort_values('z')
    print(fam, 'injury down:', ', '.join(i.source[i.z < 0].head(30)), '| up:', ', '.join(i.source[i.z > 0].tail(30)))

## 6 · PROGENy against its named sets (descriptive)

In [ ]:
FVS = []
for path, (hall, kegg) in FOOT_SETS.items():
    for fam, name in (('FOOT', path), ('HALL', hall), ('MEM', kegg)):
        if name is None:
            continue
        i = ITAB[ITAB.family.eq(fam) & ITAB.source.eq(name)]
        z = ZTAB[ZTAB.family.eq(fam) & ZTAB.source.eq(name)]
        if i.empty:
            FVS.append({'PROGENy': path, 'family': fam, 'source': name, 'note': 'not a source (too few targets or absent)'})
            continue
        r, zr = i.iloc[0], z.iloc[0] if not z.empty else None
        FVS.append({'PROGENy': path, 'family': fam, 'source': name, 'injury z': r.z, 'injury q': r.q,
                    'pairs agree': r['pairs agree'], 'z R1': r['z R1'], 'z R2': r['z R2'], 'called': r.called,
                    f'Kirita z {T_LABEL[T_STAR]}': r[f'z {t_star}'], f'Kirita q {T_LABEL[T_STAR]}': r[f'q {t_star}'],
                    'zonation category': zr.category if zr is not None else None})
FVS = pd.DataFrame(FVS)
FVS.to_csv(tables / 'foot_vs_sets.csv', index=False)
display(FVS.round(3))

## 7 · Recovery checks (Kirita 2020's main text) and TF mRNA

In [ ]:
REC = []
reg_i = ITAB[ITAB.family.eq('REG')].set_index('source')
for check, src, direction in RECOVERY:
    if src not in reg_i.index:
        REC.append({'check': check, 'source': src, 'expected': '+' if direction > 0 else '−', 'note': 'not a source'})
        continue
    r = reg_i.loc[src]
    REC.append({'check': check, 'source': src, 'expected': '+' if direction > 0 else '−', 'injury z': r.z, 'q': r.q,
                'called': bool(r.called), 'holds (q ≤ 0.05, expected sign)': bool(r.q <= ALPHA and np.sign(r.z) == direction),
                **{f'Kirita {T_LABEL[t]} z': r[f'z Kirita level {T_LABEL[t]}'] for t in TIMES}})
REC = pd.DataFrame(REC)
REC.to_csv(tables / 'recovery_checks.csv', index=False)
display(REC.round(2))

m2h = {v: k for k, v in h2m.items()}
reg_called = reg_i[reg_i.called]
tf_gene = {s: h2m.get(s) for s in reg_called.index}
mr = [(s, tf_gene[s]) for s in reg_called.index if tf_gene[s] in G91.index]
MRNA = pd.DataFrame([{'source': s, 'gene': g, 'activity z': reg_called.loc[s, 'z'], 'mRNA L': G91.loc[g, 'L'],
                      'same sign': bool(np.sign(reg_called.loc[s, 'z']) == np.sign(G91.loc[g, 'L']))} for s, g in mr])
MRNA.to_csv(tables / 'mrna_agreement.csv', index=False)
print(f'REG calls with a mapped TF gene: {len(MRNA)}; same sign as its mRNA: '
      f'{MRNA["same sign"].mean() if len(MRNA) else float("nan"):.2f}')

### 7.1 · Post hoc: threshold-free agreement per family

Added after the first run showed that very few sources pass q ≤ 0.05 in both specimens of a species while their
z rankings agree closely between mice. Spearman ρ over each family's sources between our z and the external z:
zonation (mouse mean against Kirita's sham mice, human mean against Census, mouse against human) and injury (L
against Kirita at each time point; the relabelings against Kirita as the specificity reference). Not judged.

In [ ]:
POST = []
for fam in FAMILIES:
    z = ZTAB[ZTAB.family.eq(fam)]
    m = z[[f'z {s}' for s in MOUSE_SPEC]].mean(axis=1)
    h = z[[f'z {s}' for s in HUMAN_SPEC]].mean(axis=1)
    row = {'family': fam, 'sources (Z)': len(z),
           'Z · Ctrl1A2 vs Ctrl1A4': spearmanr(z[f'z {MOUSE_SPEC[0]}'], z[f'z {MOUSE_SPEC[1]}'])[0],
           'Z · COR1 vs MED1': spearmanr(z[f'z {HUMAN_SPEC[0]}'], z[f'z {HUMAN_SPEC[1]}'])[0],
           'Z · mouse vs Kirita sham': spearmanr(m, z['z Kirita sham'], nan_policy='omit')[0],
           'Z · human vs Census': spearmanr(h, z['z Census human'], nan_policy='omit')[0],
           'Z · mouse vs human': spearmanr(m, h)[0]}
    i = ITAB[ITAB.family.eq(fam)]
    row['sources (I)'] = len(i)
    for t in TIMES:
        for vec, col in (('L', 'z'), ('R1', 'z R1'), ('R2', 'z R2')):
            row[f'I · {vec} vs Kirita {T_LABEL[t]}'] = spearmanr(i[col], i[f'z Kirita level {T_LABEL[t]}'], nan_policy='omit')[0]
    POST.append(row)
POST = pd.DataFrame(POST).set_index('family')
POST.to_csv(tables / 'agreement_post_hoc.csv')
display(POST.round(3).T)

## 8 · Figures

- 95a: zonation, mouse against human activity z for the regulons (mean over each species' two specimens).
- 95b: the scorecard by family.
- 95c: injury activity z against Kirita's at t*, per family.
- 95d: PROGENy pathways against their named Hallmark and KEGG sets: injury z, pairs and relabelings.

In [ ]:
CAT_COLOR = {'conserved': '#1b8a5a', 'reversed': '#d55e00', 'mouse only': '#2a78d6', 'human only': '#9C93D3', 'none': LIGHT}
z = ZTAB[ZTAB.family.eq('REG')].copy()
z['mouse'] = z[[f'z {s}' for s in MOUSE_SPEC]].mean(axis=1)
z['human'] = z[[f'z {s}' for s in HUMAN_SPEC]].mean(axis=1)
fig, ax = plt.subplots(figsize=(75 * MM, 70 * MM))
for cat in ['none', 'mouse only', 'human only', 'reversed', 'conserved']:
    b = z[z.category.eq(cat)]
    ax.scatter(b.mouse, b.human, s=4 if cat != 'none' else 2, color=CAT_COLOR[cat], lw=0, label=f'{cat} ({len(b)})')
for _, r in pd.concat([z[z.category.eq('conserved')].assign(m=lambda d: d.mouse.abs() + d.human.abs()).nlargest(8, 'm'),
                       z[z.category.eq('reversed')].assign(m=lambda d: d.mouse.abs() + d.human.abs()).nlargest(6, 'm'),
                       z[z.category.eq('mouse only')].assign(m=lambda d: d.mouse.abs()).nlargest(8, 'm')]).iterrows():
    ax.annotate(r.source, (r.mouse, r.human), fontsize=5, xytext=(2, 2), textcoords='offset points')
ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
ax.set_xlabel('mouse activity z (S3-high > 0)'); ax.set_ylabel('human activity z (S3-high > 0)')
ax.legend(loc='upper left', markerscale=2)
save(fig, '95a_zonation_regulons')
plt.show()

crit = LAYER_CRITERIA['Z'] + LAYER_CRITERIA['I']
fig, axes = plt.subplots(1, len(crit), figsize=(170 * MM, 40 * MM))
for ax, c in zip(axes, crit):
    vals = SCORE[c].reindex(FAMILIES)
    ax.bar(range(len(FAMILIES)), vals.fillna(0).to_numpy(), color=[FAMILY_COLOR[f] for f in FAMILIES])
    for j, v in enumerate(vals):
        if not np.isfinite(v):
            ax.plot(j, 0, 'x', color=MUTED, ms=3)
    ax.set_xticks(range(len(FAMILIES)), FAMILIES, rotation=90)
    ax.set_title(c, fontsize=6)
fig.tight_layout()
save(fig, '95b_scorecard')
plt.show()

fig, axes = plt.subplots(1, len(FAMILIES), figsize=(170 * MM, 45 * MM))
for ax, fam in zip(axes, FAMILIES):
    i = ITAB[ITAB.family.eq(fam)]
    ax.scatter(i.z[~i.called], i[f'z {t_star}'][~i.called], s=2, color=LIGHT, lw=0)
    ax.scatter(i.z[i.called], i[f'z {t_star}'][i.called], s=4, color=FAMILY_COLOR[fam], lw=0)
    ax.axhline(0, color=MUTED, lw=.5); ax.axvline(0, color=MUTED, lw=.5)
    ax.set_title(fam)
    ax.set_xlabel('injury z (ours)')
axes[0].set_ylabel(f'IRI z ({T_LABEL[T_STAR]})')
fig.tight_layout()
save(fig, '95c_injury_external')
plt.show()

fv = FVS.dropna(subset=['injury z'])
fig, ax = plt.subplots(figsize=(90 * MM, 70 * MM))
rows = list(dict.fromkeys(fv.PROGENy))
offset = {'FOOT': -.25, 'HALL': 0, 'MEM': .25}
for _, r in fv.iterrows():
    y = rows.index(r.PROGENy) + offset[r.family]
    ax.plot([r['z R1'], r['z R2']], [y, y], '|', color=MUTED, ms=4)
    ax.plot(r['injury z'], y, 'o' if r.called else 'o', ms=3.5, color=FAMILY_COLOR[r.family],
            mfc=FAMILY_COLOR[r.family] if r.called else 'white')
ax.axvline(0, color=MUTED, lw=.5)
ax.set_yticks(range(len(rows)), rows)
ax.invert_yaxis()
ax.set_xlabel('injury activity z')
handles = [plt.Line2D([], [], marker='o', ls='', color=FAMILY_COLOR[f], label=f) for f in ('FOOT', 'HALL', 'MEM')]
handles += [plt.Line2D([], [], marker='o', ls='', color=MUTED, mfc='white', label='not called'),
            plt.Line2D([], [], marker='|', ls='', color=MUTED, label='relabelings')]
ax.legend(handles=handles, loc='lower right')
save(fig, '95d_progeny_vs_sets')
plt.show()

## 9 · How the protocol was read, and the run record

In [ ]:
READING = pd.DataFrame([
    ('reading', 'Strata for the permutations: deciles of 91\'s control log2 CP10k over the layer\'s genes; genes without it form one stratum.'),
    ('reading', 'Kirita sham zonation uses 94\'s pseudobulks on 94\'s universe ∩ the zonation genes; each external vector has its own W.'),
    ('reading', 'Specificity uses the q ≤ 0.05 counts (condition and each relabeling), as frozen; pair agreement is over condition q ≤ 0.05.'),
    ('reading', 'A family criterion with fewer than 5 sources behind it is undefined (NaN), and dominance ignores it.'),
    ('post hoc', '7.1: threshold-free agreement per family (our z against external z; relabelings as the reference). Not judged.'),
    ('correction', 'G1: the protocol says 64,978 CollecTRI edges (wc -l minus the header); the file has 64,979 rows because its '
                   'last line has no newline. The 1,209 sources match.'),
], columns=['kind', 'what'])
READING.to_csv(tables / 'how_the_protocol_was_read.csv', index=False)
RUN = {'notebook': '95_pt_regulator_activity', 'logic_version': NOTEBOOK_LOGIC_VERSION, 'seed': SEED, 'guards': guard_log,
       'network sha256': SHA, 't_star': T_LABEL[T_STAR], 'sources': SOURCE_TABLE.to_dict('records'),
       'verdicts': VERD[['A', 'B', 'layer', 'verdict']].to_dict('records'),
       'seconds': round(time.perf_counter() - NOTEBOOK_START, 1)}
(output / 'run_record.json').write_text(json.dumps(RUN, indent=2, default=str))
print(json.dumps(RUN, indent=2, default=str))